In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
from astropy.stats import mad_std

def comprehensive_homogenization_diagnosis():
    """
    Diagnóstico completo del proceso de homogenización
    """
    
    # Leer archivo original
    df_original = pd.read_csv("../anac_data/Results/all_fields_gc_photometry_corrected_errors_v17.csv")
    print(f"📊 Datos originales: {len(df_original)} fuentes")
    
    # 1. Análisis de la distribución de magnitudes
    print("\n🔍 ANÁLISIS DE DISTRIBUCIÓN DE MAGNITUDES")
    print("=" * 50)
    
    splus_filters = ['F378', 'F395', 'F410', 'F430', 'F515', 'F660', 'F861']
    taylor_filters = ['umag', 'gmag', 'rmag', 'imag', 'zmag']
    
    # Crear figura para distribución de magnitudes
    fig, axes = plt.subplots(2, 1, figsize=(12, 10))
    
    # SPLUS filters
    splus_data = []
    for filtro in splus_filters:
        mag_col = f'MAG_{filtro}_3'
        if mag_col in df_original.columns:
            valid_data = df_original[mag_col][(df_original[mag_col] > 10) & (df_original[mag_col] < 30)]
            for mag in valid_data:
                splus_data.append({'Filter': filtro, 'Magnitude': mag, 'Type': 'SPLUS'})
    
    splus_df = pd.DataFrame(splus_data)
    if not splus_df.empty:
        sns.violinplot(data=splus_df, x='Filter', y='Magnitude', ax=axes[0])
        axes[0].set_title('Distribución de Magnitudes SPLUS')
        axes[0].set_ylabel('Magnitud')
    
    # Taylor filters
    taylor_data = []
    for filtro in taylor_filters:
        if filtro in df_original.columns:
            valid_data = df_original[filtro][(df_original[filtro] > 10) & (df_original[filtro] < 30)]
            for mag in valid_data:
                taylor_data.append({'Filter': filtro, 'Magnitude': mag, 'Type': 'Taylor'})
    
    taylor_df = pd.DataFrame(taylor_data)
    if not taylor_df.empty:
        sns.violinplot(data=taylor_df, x='Filter', y='Magnitude', ax=axes[1])
        axes[1].set_title('Distribución de Magnitudes Taylor')
        axes[1].set_ylabel('Magnitud')
    
    plt.tight_layout()
    plt.savefig('../anac_data/Results/homogenization_diagnosis_magnitudes.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # 2. Análisis de correlaciones entre filtros
    print("\n🔍 ANÁLISIS DE CORRELACIONES ENTRE FILTROS")
    print("=" * 50)
    
    # Crear matriz de correlación para filtros SPLUS
    splus_mag_cols = [f'MAG_{f}_3' for f in splus_filters if f'MAG_{f}_3' in df_original.columns]
    splus_corr_data = df_original[splus_mag_cols].copy()
    
    # Renombrar columnas para mejor visualización
    splus_corr_data.columns = [col.replace('MAG_', '').replace('_3', '') for col in splus_corr_data.columns]
    
    plt.figure(figsize=(10, 8))
    corr_matrix = splus_corr_data.corr()
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
                square=True, fmt='.3f', cbar_kws={'label': 'Correlación'})
    plt.title('Matriz de Correlación - Filtros SPLUS')
    plt.tight_layout()
    plt.savefig('../anac_data/Results/homogenization_diagnosis_splus_correlation.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print("📊 Matriz de correlación SPLUS:")
    print(corr_matrix.round(3))
    
    # 3. Análisis de diferencias sistemáticas
    print("\n🔍 ANÁLISIS DE DIFERENCIAS SISTEMÁTICAS")
    print("=" * 50)
    
    # Mapeo físico de filtros (basado en longitudes de onda)
    wavelength_map = {
        'F378': 378, 'F395': 395, 'F410': 410, 'F430': 430,
        'F515': 515, 'F660': 660, 'F861': 861,
        'umag': 354, 'gmag': 477, 'rmag': 622, 'imag': 763, 'zmag': 926
    }
    
    # Calcular diferencias para cada posible mapeo
    possible_mappings = [
        {'F378': 'umag', 'F395': 'umag', 'F410': 'umag', 'F430': 'umag', 'F515': 'gmag', 'F660': 'rmag', 'F861': 'imag'},
        {'F378': 'umag', 'F395': 'umag', 'F410': 'gmag', 'F430': 'gmag', 'F515': 'gmag', 'F660': 'rmag', 'F861': 'imag'},
        {'F378': 'umag', 'F395': 'umag', 'F410': 'umag', 'F430': 'gmag', 'F515': 'gmag', 'F660': 'rmag', 'F861': 'imag'}
    ]
    
    mapping_results = []
    
    for mapping_idx, filter_mapping in enumerate(possible_mappings):
        mapping_offsets = {}
        mapping_uncertainties = {}
        
        for splus_filt, taylor_filt in filter_mapping.items():
            splus_col = f'MAG_{splus_filt}_3'
            
            if splus_col not in df_original.columns or taylor_filt not in df_original.columns:
                continue
            
            # Filtrar datos válidos
            mask = (
                df_original[splus_col].notna() & 
                df_original[taylor_filt].notna() &
                (df_original[splus_col] < 90) & (df_original[taylor_filt] < 90) &
                (df_original[splus_col] > 10) & (df_original[taylor_filt] > 10)
            )
            
            valid_data = df_original.loc[mask]
            if len(valid_data) < 10:
                continue
            
            # Calcular diferencia
            differences = valid_data[taylor_filt] - valid_data[splus_col]
            valid_diff = differences[np.isfinite(differences)]
            
            if len(valid_diff) == 0:
                continue
                
            median_offset = np.median(valid_diff)
            mad_uncertainty = mad_std(valid_diff)
            
            mapping_offsets[splus_filt] = median_offset
            mapping_uncertainties[splus_filt] = mad_uncertainty
            
            mapping_results.append({
                'Mapping': mapping_idx,
                'SPLUS_Filter': splus_filt,
                'Taylor_Filter': taylor_filt,
                'Wavelength_SPLUS': wavelength_map[splus_filt],
                'Wavelength_Taylor': wavelength_map[taylor_filt],
                'Wavelength_Diff': abs(wavelength_map[splus_filt] - wavelength_map[taylor_filt]),
                'Offset': median_offset,
                'Uncertainty': mad_uncertainty,
                'N_Sources': len(valid_diff)
            })
    
    mapping_df = pd.DataFrame(mapping_results)
    
    # Graficar resultados de diferentes mapeos
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Gráfico 1: Offsets vs longitud de onda para cada mapeo
    for mapping_idx in mapping_df['Mapping'].unique():
        mapping_data = mapping_df[mapping_df['Mapping'] == mapping_idx]
        axes[0,0].errorbar(mapping_data['Wavelength_SPLUS'], mapping_data['Offset'], 
                          yerr=mapping_data['Uncertainty'], fmt='o-', 
                          label=f'Mapping {mapping_idx}', capsize=5)
    
    axes[0,0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
    axes[0,0].set_xlabel('Longitud de Onda SPLUS (nm)')
    axes[0,0].set_ylabel('Offset (Taylor - SPLUS)')
    axes[0,0].set_title('Offsets vs Longitud de Onda por Mapeo')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # Gráfico 2: Diferencia de longitud de onda vs offset
    scatter = axes[0,1].scatter(mapping_df['Wavelength_Diff'], mapping_df['Offset'], 
                               c=mapping_df['Uncertainty'], cmap='viridis', s=100, alpha=0.7)
    axes[0,1].set_xlabel('Diferencia de Longitud de Onda (nm)')
    axes[0,1].set_ylabel('Offset (mag)')
    axes[0,1].set_title('Offset vs Diferencia de Longitud de Onda')
    plt.colorbar(scatter, ax=axes[0,1], label='Incertidumbre (MAD)')
    axes[0,1].grid(True, alpha=0.3)
    
    # Gráfico 3: Heatmap de mapeos óptimos
    pivot_data = mapping_df.pivot_table(values='Offset', index='SPLUS_Filter', 
                                       columns='Taylor_Filter', aggfunc='mean')
    sns.heatmap(pivot_data, annot=True, cmap='coolwarm', center=0, 
                ax=axes[1,0], cbar_kws={'label': 'Offset (mag)'})
    axes[1,0].set_title('Offset Promedio por Combinación de Filtros')
    
    # Gráfico 4: Calidad del mapeo (incertidumbre)
    pivot_uncertainty = mapping_df.pivot_table(values='Uncertainty', index='SPLUS_Filter', 
                                              columns='Taylor_Filter', aggfunc='mean')
    sns.heatmap(pivot_uncertainty, annot=True, cmap='viridis', 
                ax=axes[1,1], cbar_kws={'label': 'Incertidumbre (MAD)'})
    axes[1,1].set_title('Incertidumbre por Combinación de Filtros')
    
    plt.tight_layout()
    plt.savefig('../anac_data/Results/homogenization_diagnosis_mappings.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # 4. Análisis por campo
    print("\n🔍 ANÁLISIS DE VARIACIÓN POR CAMPO")
    print("=" * 50)
    
    if 'FIELD' in df_original.columns:
        fields = df_original['FIELD'].unique()
        print(f"📁 Campos encontrados: {len(fields)}")
        
        field_analysis = []
        
        for field in fields[:10]:  # Analizar solo primeros 10 campos para no saturar
            field_data = df_original[df_original['FIELD'] == field]
            
            for splus_filt in splus_filters:
                splus_col = f'MAG_{splus_filt}_3'
                if splus_col not in field_data.columns:
                    continue
                
                # Buscar el Taylor filter más cercano
                splus_wl = wavelength_map[splus_filt]
                closest_taylor = min(taylor_filters, 
                                   key=lambda x: abs(wavelength_map[x] - splus_wl))
                
                if closest_taylor not in field_data.columns:
                    continue
                
                # Calcular offset para este campo
                mask = (
                    field_data[splus_col].notna() & 
                    field_data[closest_taylor].notna() &
                    (field_data[splus_col] < 90) & (field_data[closest_taylor] < 90) &
                    (field_data[splus_col] > 10) & (field_data[closest_taylor] > 10)
                )
                
                valid_field_data = field_data.loc[mask]
                if len(valid_field_data) < 5:
                    continue
                
                differences = valid_field_data[closest_taylor] - valid_field_data[splus_col]
                valid_diff = differences[np.isfinite(differences)]
                
                if len(valid_diff) == 0:
                    continue
                    
                median_offset = np.median(valid_diff)
                
                field_analysis.append({
                    'Field': field,
                    'SPLUS_Filter': splus_filt,
                    'Taylor_Filter': closest_taylor,
                    'Offset': median_offset,
                    'N_Sources': len(valid_diff)
                })
        
        if field_analysis:
            field_df = pd.DataFrame(field_analysis)
            
            plt.figure(figsize=(12, 8))
            for filt in splus_filters:
                filt_data = field_df[field_df['SPLUS_Filter'] == filt]
                if not filt_data.empty:
                    plt.plot(filt_data['Field'], filt_data['Offset'], 'o-', label=filt)
            
            plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
            plt.xlabel('Campo')
            plt.ylabel('Offset (Taylor - SPLUS)')
            plt.title('Variación de Offsets por Campo')
            plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.xticks(rotation=45)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig('../anac_data/Results/homogenization_diagnosis_field_variation.png', 
                       dpi=300, bbox_inches='tight')
            plt.close()
    
    # 5. Recomendaciones basadas en el análisis
    print("\n🎯 RECOMENDACIONES BASADAS EN EL DIAGNÓSTICO")
    print("=" * 50)
    
    # Encontrar el mejor mapeo (menor incertidumbre promedio)
    best_mapping = mapping_df.groupby('Mapping').agg({
        'Uncertainty': 'mean',
        'Offset': lambda x: np.mean(np.abs(x)),
        'N_Sources': 'sum'
    }).round(3)
    
    best_mapping['Score'] = best_mapping['Uncertainty'] + best_mapping['Offset']
    best_mapping_idx = best_mapping['Score'].idxmin()
    
    print(f"✅ Mejor mapeo: #{best_mapping_idx}")
    print(f"   - Incertidumbre promedio: {best_mapping.loc[best_mapping_idx, 'Uncertainty']:.3f} mag")
    print(f"   - Offset absoluto promedio: {best_mapping.loc[best_mapping_idx, 'Offset']:.3f} mag")
    print(f"   - Total de fuentes: {best_mapping.loc[best_mapping_idx, 'N_Sources']}")
    
    best_mapping_filters = mapping_df[mapping_df['Mapping'] == best_mapping_idx][['SPLUS_Filter', 'Taylor_Filter']]
    print("\n📊 Mejor combinación de filtros:")
    for _, row in best_mapping_filters.iterrows():
        print(f"   {row['SPLUS_Filter']} → {row['Taylor_Filter']}")
    
    return best_mapping_idx, mapping_df

def apply_best_homogenization(best_mapping_idx, mapping_df):
    """
    Aplica la homogenización usando el mejor mapeo encontrado
    """
    print(f"\n🔄 APLICANDO MEJOR HOMOGENIZACIÓN (Mapeo #{best_mapping_idx})")
    print("=" * 50)
    
    # Leer archivo original
    df = pd.read_csv("../anac_data/Results/all_fields_gc_photometry_corrected_errors_v17.csv")
    
    # Obtener el mejor mapeo
    best_mapping_data = mapping_df[mapping_df['Mapping'] == best_mapping_idx]
    filter_mapping = {}
    offsets = {}
    
    for _, row in best_mapping_data.iterrows():
        splus_col = f"MAG_{row['SPLUS_Filter']}_3"
        filter_mapping[splus_col] = row['Taylor_Filter']
        offsets[row['SPLUS_Filter']] = row['Offset']
    
    print("📊 Aplicando offsets:")
    for filtro, offset in offsets.items():
        mag_column = f"MAG_{filtro}_3"
        if mag_column in df.columns:
            df[mag_column] = df[mag_column] + offset
            print(f"  ✅ {mag_column}: +{offset:+.3f} mag")
    
    # Guardar archivo homogenizado
    output_path = f"../anac_data/Results/all_fields_gc_photometry_optimized_mapping_{best_mapping_idx}.csv"
    df.to_csv(output_path, index=False)
    print(f"\n💾 Archivo homogenizado guardado: {output_path}")
    
    return output_path

if __name__ == "__main__":
    print("🎯 DIAGNÓSTICO COMPLETO DE HOMOGENIZACIÓN")
    print("=" * 70)
    print("Este análisis identificará el mejor mapeo de filtros y")
    print("detectará problemas sistemáticos en la homogenización.")
    print("=" * 70)
    
    # Ejecutar diagnóstico completo
    best_mapping_idx, mapping_df = comprehensive_homogenization_diagnosis()
    
    # Aplicar el mejor mapeo encontrado
    output_path = apply_best_homogenization(best_mapping_idx, mapping_df)
    
    print("\n✅ DIAGNÓSTICO COMPLETADO")
    print("=" * 70)
    print("📊 Archivos generados:")
    print("   - Results/homogenization_diagnosis_magnitudes.png")
    print("   - Results/homogenization_diagnosis_splus_correlation.png") 
    print("   - Results/homogenization_diagnosis_mappings.png")
    print("   - Results/homogenization_diagnosis_field_variation.png")
    print(f"   - {output_path}")
    print("\n🔍 Revisa los gráficos para entender los problemas y soluciones.")

🎯 DIAGNÓSTICO COMPLETO DE HOMOGENIZACIÓN
Este análisis identificará el mejor mapeo de filtros y
detectará problemas sistemáticos en la homogenización.
📊 Datos originales: 3799 fuentes

🔍 ANÁLISIS DE DISTRIBUCIÓN DE MAGNITUDES

🔍 ANÁLISIS DE CORRELACIONES ENTRE FILTROS
📊 Matriz de correlación SPLUS:
       F378   F395   F410   F430   F515   F660   F861
F378  1.000  0.346  0.366  0.377  0.459  0.498  0.500
F395  0.346  1.000  0.371  0.390  0.439  0.494  0.491
F410  0.366  0.371  1.000  0.429  0.485  0.555  0.545
F430  0.377  0.390  0.429  1.000  0.513  0.574  0.570
F515  0.459  0.439  0.485  0.513  1.000  0.752  0.744
F660  0.498  0.494  0.555  0.574  0.752  1.000  0.920
F861  0.500  0.491  0.545  0.570  0.744  0.920  1.000

🔍 ANÁLISIS DE DIFERENCIAS SISTEMÁTICAS

🔍 ANÁLISIS DE VARIACIÓN POR CAMPO
📁 Campos encontrados: 18

🎯 RECOMENDACIONES BASADAS EN EL DIAGNÓSTICO
✅ Mejor mapeo: #1
   - Incertidumbre promedio: 0.640 mag
   - Offset absoluto promedio: 0.603 mag
   - Total de fuentes: 18